# Table of Contents

1. [Life of a Query](#life-of-a-query)
2. [Databases Indexes](#database-indexes)
3. [Query Tips (Pruning)](#avoid-select---read-only-what-you-need)


# Life of a Query 

![Life of a Query](./images/life_of_query.png)

## Query Flow
1. **Query issued** → you submit SQL.  
2. **Parsing** → tokenize, syntax/semantic checks, permissions.  
3. **Planning/Optimization** → generate multiple plans, estimate cost (I/O, CPU, memory), pick cheapest.  
4. **Execution** → execution engine runs chosen plan against the storage engine and returns results.

## Core DBMS Components
| Component | What it does |
|---|---|
| **Transport System** | Receives requests from clients. |
| **Query Parser** | Breaks SQL into tokens, validates schema names & privileges. |
| **Query Optimizer** | Explores strategies (joins, scans, index usage) and chooses the lowest-cost plan. |
| **Execution Engine** | Carries out the operations (scans, joins, sorts) in the plan. |
| **Storage Engine** | Reads/writes data pages on disk/SSD and manages buffers. |

## Seeing the Plan with `EXPLAIN`
Use `EXPLAIN` to view the plan (before running) or to understand a slow query (after, with variants like `EXPLAIN ANALYZE`).

```sql
EXPLAIN SELECT * FROM customer;
```

- **Plan:** Sequential (full table) scan  
- **Typical outputs:** startup cost, total cost, estimated rows, row width

![Understanding Query Performance](./images/explain_query.png)

```sql
EXPLAIN SELECT * 
FROM customer 
WHERE customer_id = 3;
```

- **Plan:** Index scan on `customer_id` (uses the index instead of scanning the whole table)  
- **Why faster:** Jumps directly to the matching row(s), reducing I/O → lower **total cost**

## Indexes (Why They Matter)
An **index** is a side data structure that speeds up lookups (like a book’s index).  
- Great for equality filters (`WHERE id = …`) and many range queries  
- Trade-off: extra storage + slower writes (index maintenance)



# Database Indexes

Indexes are **separate data structures** that store ordered information about one or more columns of a table, helping the DBMS quickly locate rows **without scanning the entire table**.

---

## 📘 Book Analogy
Just like a **book index** lets you find topics without reading every page, a **database index** helps the DBMS find specific rows efficiently.

![Book Index Analogy](./images/index.png)

---

## 🧩 Index Data Structure
An index isn’t stored as one big sequential table. It’s divided into **sorted blocks** that are **doubly linked** for forward/backward traversal.  
- Physical location of blocks doesn’t matter.  
- Inserts/deletes update only the affected block(s).

![Index Data Structure](./images/index_data_structure.png)

---

## 🌳 Balanced Search Tree (B-Tree)
Most row-store DBs implement indexes as a **B-Tree**:
- **Root → Branch → Leaf nodes**
- Leaves hold the indexed values + row addresses
- Search descends the tree to the correct leaf, then reads matching entries (and possibly adjacent leaves if values repeat)

Example:
```sql
SELECT *
FROM orders
WHERE country = 'Canada';
```

![Balanced Search Tree](./images/BST_B_TREE.png)

---

## ⏱️ Time Complexity
- **Tree traversal:** `O(log n)` (fast due to balance)
- **Chain traversal across leaves:** needed when many rows share the same indexed value; can get expensive → optimizer may prefer a **Sequential Scan**

![B-Tree Time & Repeats](./images/time_complexity.png)

---

## 🧮 Example: Adding an Index

**Without index**
```sql
EXPLAIN SELECT * FROM payment WHERE rental_id = 1;
```
➡️ Plan shows a **Sequential (Full Table) Scan**

**Create index & rerun**
```sql
CREATE INDEX rental_idx ON payment (rental_id);

EXPLAIN SELECT * FROM payment WHERE rental_id = 1;
```
➡️ Plan shows an **Index Scan** with much lower cost (often >30× faster)

![Index Example](./images/index_.png)

---

## 🗄️ Columnar Stores (Redshift/BigQuery)
Column warehouses don’t use separate B-Tree indexes for scans. Instead, you define **Sort/Cluster Keys**:
- Data is **physically sorted** on disk by the key columns
- Prunes large swaths of data during scans

![Columnar Storage](./images/columnar_storage.png)

---

## ✅ Takeaways
- Indexes speed up filters/joins/orderings on chosen columns.  
- Too many indexes slow **writes** (each index must be maintained).  
- In column stores, use **sort/cluster keys** for the same pruning effect.


# Avoid `SELECT *` — Read Only What You Need

![Avoid Select *](./images/avoid_select_*.png)

Running `SELECT *` (no `WHERE`, all columns) forces the engine to:
- **Scan the entire table**
- **Transfer all bytes** from disk
- **Cost more** on pay-as-you-go warehouses (I/O + compute)

## ✅ Pruning: Exclude Irrelevant Data

![Query Tips](./images/query_tips.png)

### 1) Row-based pruning
Filter rows with predicates; combine with indexes (row stores) or sort/cluster keys (column stores).

```sql
-- Row-based pruning
SELECT *
FROM payment
WHERE rental_id = 1;
```

### 2) Column-based pruning
Select only the columns you need so the warehouse reads fewer columns from disk.

```sql
-- Column-based pruning
SELECT customer_id, rental_id
FROM payment;
```

### 3) Partition pruning
Partition by a key (e.g., date); the engine scans only relevant partitions.

```sql
-- Example idea (syntax varies by system)
SELECT *
FROM orders
WHERE order_date = DATE '2024-04-01'
  AND country = 'USA';
```

> In BigQuery/Redshift/Snowflake: use **partitioning** + **clustering/sort keys** so the optimizer can skip entire partitions and most blocks inside them.

## 🧠 Rules of Thumb
- Never `SELECT *` on large tables in prod.
- Always add **selective WHERE** predicates.
- Project only needed columns.
- Use **partitioning** and **cluster/sort keys** aligned with your most common filters.